# 02. 결과 분석

실험 완료 후 `results/` 폴더의 CSV를 불러와 논문 Table 형식으로 분석합니다.

- Table I : 주요 성능 비교 (실험 1)
- Table III: Ablation - Score 성분 (실험 2)
- Table IV : Ablation - 레이어 할당 (실험 3)
- Table V  : 예산 민감도 (실험 4)
- 가설 H1 / H2 / H3 검증 요약

In [ ]:
import sys, os, glob
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np

RESULTS_DIR = 'results'

def load_latest_csv(prefix):
    """results/ 에서 prefix로 시작하는 가장 최근 CSV 로드."""
    files = sorted(glob.glob(f'{RESULTS_DIR}/{prefix}*.csv'))
    if not files:
        print(f'  [없음] {prefix}*.csv - 실험을 먼저 실행하세요')
        return pd.DataFrame()
    print(f'  [로드] {files[-1]}')
    return pd.read_csv(files[-1])

print('환경 준비 완료')

## Table I: 주요 성능 비교 (실험 1-A)

In [ ]:
df1 = load_latest_csv('exp1_qwen3-4b_full')

# 논문 기대값 (Budget=20%)
PAPER = {
    'FULLKV': 37.5, 'STREAMING': 26.2, 'H2O': 31.2,
    'SNAPKV': 33.8, 'PYRAMIDKV': 34.5, 'ADAKV': 35.1, 'OURS': 36.6,
}

if not df1.empty:
    print('=== Table I: 주요 성능 비교 (Budget=20%) ===')
    print(df1.to_string(index=False))
    print()

    if 'Avg' in df1.columns and 'Method' in df1.columns:
        print('=== 논문 기대값 vs 실측값 (허용범위 ±0.5p) ===')
        header = f'{"Method":<15} {"논문":>6} {"실측":>6} {"차이":>6}  OK?'
        print(header)
        print('-' * len(header))
        for _, row in df1.iterrows():
            method   = str(row['Method'])
            actual   = float(row['Avg'])
            expected = PAPER.get(method, None)
            if expected is not None:
                diff = actual - expected
                ok   = 'OK' if abs(diff) <= 0.5 else 'CHECK'
                print(f'{method:<15} {expected:>6.1f} {actual:>6.1f} {diff:>+6.1f}  {ok}')

## Table III: Ablation - Score 성분 (실험 2)

In [ ]:
df2 = load_latest_csv('exp2_score_ablation_qwen3-4b')

if not df2.empty:
    print('=== Table III: Ablation - Hybrid Score 성분 ===')
    print(df2.to_string(index=False))
    print()

    # H1 가설 검증
    if 'Avg' in df2.columns and 'Condition' in df2.columns:
        full_rows = df2[df2['Condition'].str.contains('Full', na=False)]
        if not full_rows.empty:
            full_score  = float(full_rows['Avg'].iloc[0])
            other_rows  = df2[~df2['Condition'].str.contains('Full', na=False)]
            all_lower   = (other_rows['Avg'] < full_score).all()
            worst_cond  = df2.loc[df2['Avg'].idxmin(), 'Condition']
            worst_score = float(df2['Avg'].min())
            print(f'H1 검증: Full(4신호) = {full_score:.1f}')
            print(f'  4신호 > 모든 단일 제거: {"H1 지지 (OK)" if all_lower else "H1 기각 (CHECK)"}')
            print(f'  가장 중요 신호: {worst_cond} (제거시 {worst_score:.1f}, 하락폭 최대)')

## Table IV: Ablation - 레이어 할당 (실험 3)

In [ ]:
df3 = load_latest_csv('exp3_allocation_qwen3-4b')

if not df3.empty:
    print('=== Table IV: Ablation - 레이어 할당 전략 ===')
    print(df3.to_string(index=False))
    print()

    if 'Avg_Score' in df3.columns and 'Strategy' in df3.columns:
        ent_rows = df3[df3['Strategy'].str.contains('Entropy', na=False)]
        if not ent_rows.empty:
            ent_score  = float(ent_rows['Avg_Score'].iloc[0])
            other_rows = df3[~df3['Strategy'].str.contains('Entropy', na=False)]
            all_lower  = (other_rows['Avg_Score'] < ent_score).all()
            print(f'H2 검증: Entropy-Driven = {ent_score:.1f}')
            print(f'  Entropy > 모든 대안 전략: {"H2 지지 (OK)" if all_lower else "H2 기각 (CHECK)"}')
            for _, row in other_rows.iterrows():
                delta = float(row['Avg_Score']) - ent_score
                print(f'    vs {row["Strategy"]}: {delta:+.1f}')

## Table V: 예산 민감도 (실험 4)

In [ ]:
df4 = load_latest_csv('exp4_budget_sensitivity_qwen3-4b')

if not df4.empty:
    print('=== Table V: 예산 민감도 ===')
    print(df4.to_string(index=False))
    print()

    if 'Pct_of_FullKV_%' in df4.columns and 'Budget' in df4.columns:
        at_100 = df4[df4['Pct_of_FullKV_%'] >= 100.0]
        if not at_100.empty:
            min_budget = at_100['Budget'].iloc[0]
            print(f'핵심: {min_budget} 예산에서 Full KV 동등 달성')
            print(f'  논문 목표: 30% 예산  ->  {"OK" if "30" in str(min_budget) else "확인 필요"}')
        else:
            print('아직 Full KV 동등 예산 미달성 (더 높은 budget 실험 필요)')

## 교차 모델 (실험 1-B, 1-C): H3 검증

In [ ]:
# 논문 Table II 기대값
CROSS_PAPER = {
    'qwen3-4b':  {'FullKV': 37.5, 'AdaKV': 35.1, 'Ours': 36.6},
    'phi-3-mini': {'FullKV': 39.2, 'AdaKV': 36.8, 'Ours': 38.6},
    'gemma-2-2b': {'FullKV': 42.1, 'AdaKV': 39.7, 'Ours': 41.0},
}

print('=== Table II: 교차 아키텍처 결과 ===')
print()

models_to_check = ['qwen3-4b', 'phi-3-mini', 'gemma-2-2b']
for model_key in models_to_check:
    prefix = f'exp1_{model_key}'
    df_m = load_latest_csv(prefix)
    expected = CROSS_PAPER.get(model_key, {})

    print(f'--- {model_key} ---')
    print(f'  Expected: FullKV={expected.get("FullKV","?")}, AdaKV={expected.get("AdaKV","?")}, Ours={expected.get("Ours","?")}')
    if not df_m.empty and 'Avg' in df_m.columns:
        for _, row in df_m.iterrows():
            print(f'  Actual  : {row["Method"]}={row["Avg"]:.1f}')
    print()

## 가설 검증 최종 요약

In [ ]:
print('=' * 60)
print('가설 검증 최종 요약')
print('=' * 60)
print()

# H1
print('H1: 하이브리드 점수화(4-signal)는 단일 메트릭보다 유의미하게 우수')
df2 = load_latest_csv('exp2_score_ablation_qwen3-4b')
if not df2.empty and 'Avg' in df2.columns:
    full_rows   = df2[df2['Condition'].str.contains('Full', na=False)]
    other_rows  = df2[~df2['Condition'].str.contains('Full', na=False)]
    if not full_rows.empty:
        full_s = float(full_rows['Avg'].iloc[0])
        h1_ok  = (other_rows['Avg'] < full_s).all()
        print(f'  -> {"H1 지지" if h1_ok else "H1 기각"}')
else:
    print('  -> 실험 2 미실행')

print()

# H2
print('H2: 동적 entropy 할당이 균일 압축보다 성능 향상')
df3 = load_latest_csv('exp3_allocation_qwen3-4b')
if not df3.empty and 'Avg_Score' in df3.columns:
    ent_rows   = df3[df3['Strategy'].str.contains('Entropy', na=False)]
    other_rows = df3[~df3['Strategy'].str.contains('Entropy', na=False)]
    if not ent_rows.empty:
        ent_s = float(ent_rows['Avg_Score'].iloc[0])
        h2_ok = (other_rows['Avg_Score'] < ent_s).all()
        print(f'  -> {"H2 지지" if h2_ok else "H2 기각"}')
else:
    print('  -> 실험 3 미실행')

print()

# H3
print('H3: 다양한 SLM 아키텍처에서 일반화 성능')
cross_models = ['phi-3-mini', 'gemma-2-2b']
done = [m for m in cross_models if not load_latest_csv(f'exp1_{m}').empty]
print(f'  -> 완료 모델: {done if done else "없음 (실험 1-B/C 미실행)"}')

print()
print('=' * 60)